## Module 2: Code generation (with test run and self-fix)

In [1]:
import re
import subprocess
import sys
import tempfile
import os
import ollama


In [2]:
MODEL = "gemma4"

SYSTEM = (
    "You are an expert programmer. Write correct, clean, runnable Python code. "
    "Return the code in ONE ```python block, then a short explanation."
)


def ask(messages):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            *messages
        ],
        options={"temperature": 0}
    )
    return response["message"]["content"]


def extract_code(text):
    match = re.search(r"```python\s*(.*?)```", text, re.S)
    return match.group(1).strip() if match else None


def run_python(code, timeout=10):
    # WARNING: subprocess execution is not a secure sandbox.
    # Run only trusted code. Use an isolated container for untrusted code.
    path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".py",
            encoding="utf-8",
            delete=False
        ) as file:
            file.write(code)
            path = file.name

        result = subprocess.run(
            [sys.executable, "-I", path],
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=tempfile.gettempdir()
        )

        output = (result.stdout + result.stderr).strip()
        return result.returncode == 0, output

    except subprocess.TimeoutExpired:
        return False, f"Timed out after {timeout} seconds"

    except Exception as error:
        return False, str(error)

    finally:
        if path and os.path.exists(path):
            os.remove(path)


def generate_code(request, max_fixes=2):
    messages = [
        {"role": "user", "content": request}
    ]

    answer = ask(messages)

    for attempt in range(max_fixes + 1):
        code = extract_code(answer)

        if not code:
            print("The model did not return a Python code block.")
            return answer

        ok, output = run_python(code)

        if ok:
            return (
                answer
                + "\n\nTest run passed."
                + f"\nOutput:\n{output[:1000]}"
            )

        if attempt == max_fixes:
            return (
                answer
                + "\n\nExecution failed after repair attempts."
                + f"\nLast error:\n{output[:1000]}"
            )

        print(
            f"Attempt {attempt + 1} failed. "
            "Asking Ollama to fix the code..."
        )

        messages.extend([
            {"role": "assistant", "content": answer},
            {
                "role": "user",
                "content": (
                    "The generated Python code failed when executed.\n"
                    f"Error/output:\n{output[:3000]}\n"
                    "Return the complete corrected code in one "
                    "```python block, followed by a short explanation."
                )
            }
        ])

        answer = ask(messages)

    return answer


if __name__ == "__main__":
    request = (
        "Write a Python function to check if a number is prime "
        "and print all prime numbers up to 50."
    )

    print(generate_code(request))


```python
import math

def is_prime(n: int) -> bool:
    """
    Checks if a given integer n is a prime number.
    A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself.
    """
    if n <= 1:
        return False
    if n == 2:
        return True
    # Optimization: Check divisibility by 2 first
    if n % 2 == 0:
        return False
    
    # Check for odd divisors up to the square root of n
    # We only need to check up to sqrt(n) because if n has a divisor larger than sqrt(n), 
    # it must also have a divisor smaller than sqrt(n).
    limit = int(math.sqrt(n)) + 1
    for i in range(3, limit, 2):
        if n % i == 0:
            return False
            
    return True

def find_and_print_primes(limit: int):
    """
    Iterates through numbers up to the specified limit and prints all prime numbers.
    """
    print(f"--- Prime Numbers up to {limit} ---")
    primes = []
    for number in range(2, limit + 1):
        if i

In [3]:

import ollama
import re
import json

MODEL = "gemma4"

# Each task includes test cases and expected outputs.
tasks = [
    {
        "task": "Return the sum of two numbers.",
        "tests": [
            {"input": [2, 3], "expected": 5},
            {"input": [-1, 1], "expected": 0}
        ]
    },
    {
        "task": "Return True if a number is even, otherwise False.",
        "tests": [
            {"input": [4], "expected": True},
            {"input": [7], "expected": False}
        ]
    },
    {
        "task": "Return the largest of three numbers.",
        "tests": [
            {"input": [2, 8, 5], "expected": 8},
            {"input": [-1, -3, -2], "expected": -1}
        ]
    },
    {
        "task": "Return a string reversed.",
        "tests": [
            {"input": ["hello"], "expected": "olleh"},
            {"input": ["Python"], "expected": "nohtyP"}
        ]
    },
    {
        "task": "Return the factorial of a non-negative integer.",
        "tests": [
            {"input": [5], "expected": 120},
            {"input": [0], "expected": 1}
        ]
    },
    {
        "task": "Return True if a string is a palindrome.",
        "tests": [
            {"input": ["madam"], "expected": True},
            {"input": ["hello"], "expected": False}
        ]
    },
    {
        "task": "Return a list containing only even numbers.",
        "tests": [
            {"input": [[1, 2, 3, 4]], "expected": [2, 4]},
            {"input": [[7, 9]], "expected": []}
        ]
    },
    {
        "task": "Return the number of vowels in a string.",
        "tests": [
            {"input": ["hello"], "expected": 2},
            {"input": ["sky"], "expected": 0}
        ]
    },
    {
        "task": "Return the nth Fibonacci number with F(0)=0 and F(1)=1.",
        "tests": [
            {"input": [6], "expected": 8},
            {"input": [0], "expected": 0}
        ]
    },
    {
        "task": "Return the sum of all numbers in a list.",
        "tests": [
            {"input": [[1, 2, 3]], "expected": 6},
            {"input": [[]], "expected": 0}
        ]
    }
]

def generate_function(task):
    prompt = f"""
Write one Python function for this task:
{task}

Requirements:
- Return only a Python code block.
- Use the function name solve.
- Do not use input() or print().
- Return the result from solve().
"""

    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    text = response["message"]["content"]
    match = re.search(r"```(?:python)?\s*(.*?)```", text, re.S)

    if not match:
        raise ValueError("No Python code block returned.")

    return match.group(1)

# NOTE: This evaluator executes generated code.
# Only use it with a trusted local model and safe tasks.
# For untrusted code, use a properly restricted sandbox.

passed_tests = 0
total_tests = 0
task_results = []

for index, item in enumerate(tasks, start=1):
    try:
        code = generate_function(item["task"])
        namespace = {}

        # Only run generated code you trust.
        exec(code, {"__builtins__": __builtins__}, namespace)
        solve = namespace["solve"]

        task_passed = 0

        for test in item["tests"]:
            total_tests += 1

            try:
                result = solve(*test["input"])
                passed = result == test["expected"]

                if passed:
                    passed_tests += 1
                    task_passed += 1

            except Exception as error:
                print(f"Task {index} test error: {error}")

        task_results.append({
            "task": index,
            "passed": task_passed,
            "total": len(item["tests"])
        })

        print(
            f"Task {index}: {task_passed}/"
            f"{len(item['tests'])} tests passed"
        )

    except Exception as error:
        total_tests += len(item["tests"])
        print(f"Task {index} failed: {error}")

accuracy = (
    100 * passed_tests / total_tests
    if total_tests else 0
)

print("\n======= CODE MODEL EVALUATION =======")
print("Model:", MODEL)
print(f"Tests passed: {passed_tests}/{total_tests}")
print(f"Test pass rate: {accuracy:.2f}%")
print("Task results:", json.dumps(task_results, indent=2))
print("This is a small custom benchmark, not a universal accuracy score.")


Task 1: 2/2 tests passed
Task 2: 2/2 tests passed
Task 3: 2/2 tests passed
Task 4: 2/2 tests passed
Task 5: 2/2 tests passed
Task 6: 2/2 tests passed
Task 7: 2/2 tests passed
Task 8: 2/2 tests passed
Task 9: 2/2 tests passed
Task 10: 2/2 tests passed

======= CODE MODEL EVALUATION =======
Model: gemma4
Tests passed: 20/20
Test pass rate: 100.00%
Task results: [
  {
    "task": 1,
    "passed": 2,
    "total": 2
  },
  {
    "task": 2,
    "passed": 2,
    "total": 2
  },
  {
    "task": 3,
    "passed": 2,
    "total": 2
  },
  {
    "task": 4,
    "passed": 2,
    "total": 2
  },
  {
    "task": 5,
    "passed": 2,
    "total": 2
  },
  {
    "task": 6,
    "passed": 2,
    "total": 2
  },
  {
    "task": 7,
    "passed": 2,
    "total": 2
  },
  {
    "task": 8,
    "passed": 2,
    "total": 2
  },
  {
    "task": 9,
    "passed": 2,
    "total": 2
  },
  {
    "task": 10,
    "passed": 2,
    "total": 2
  }
]
This is a small custom benchmark, not a universal accuracy score.


In [4]:

import ollama
import time
import pandas as pd

MODEL = "gemma4"

tests = [
    {
        "question": "What is 2 + 2? Answer only the number.",
        "expected": "4"
    },
    {
        "question": "How many days are in a week? Answer only the number.",
        "expected": "7"
    },
    {
        "question": "What is the capital of France? Answer only the city.",
        "expected": "Paris"
    },
    {
        "question": "What is 10 * 5? Answer only the number.",
        "expected": "50"
    },
    {
        "question": "Which programming language uses Pandas? Answer with the language.",
        "expected": "Python"
    }
]

results = []

for test in tests:
    start = time.perf_counter()

    try:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "user", "content": test["question"]}
            ],
            options={"temperature": 0}
        )

        answer = response["message"]["content"].strip()
        elapsed = time.perf_counter() - start

        # Simple exact/substring check for short answers
        expected = test["expected"].lower()
        passed = expected in answer.lower()

        results.append({
            "Question": test["question"],
            "Expected": test["expected"],
            "Model Answer": answer,
            "Passed": passed,
            "Time (seconds)": round(elapsed, 2)
        })

    except Exception as error:
        results.append({
            "Question": test["question"],
            "Expected": test["expected"],
            "Model Answer": str(error),
            "Passed": False,
            "Time (seconds)": None
        })

df = pd.DataFrame(results)
display(df)

score = df["Passed"].mean() * 100

print(f"\nModel: {MODEL}")
print(f"Test pass rate: {score:.2f}%")
print(f"Tests passed: {df['Passed'].sum()}/{len(df)}")

print("\nAverage response time:")
print(df["Time (seconds)"].mean(), "seconds")


,Question,Expected,Model Answer,Passed,Time (seconds)
0,What is 2 + 2? Answer only the number.,4,4,True,2.96
1,How many days are in a week? Answer only the n...,7,7,True,2.76
2,What is the capital of France? Answer only the...,Paris,Paris,True,3.04
3,What is 10 * 5? Answer only the number.,50,50,True,3.93
4,Which programming language uses Pandas? Answer...,Python,Python,True,5.40



Model: gemma4
Test pass rate: 100.00%
Tests passed: 5/5

Average response time:
3.618 seconds
